# Conv2d forward in NumPy, no loops

Implement the forward pass of a 2D convolution layer (PyTorch's `F.conv2d` with `groups=1`) in NumPy, without Python loops.

```python
def conv2d(x, weight, bias=None, stride=1, padding=0, dilation=1):
    ...
    return out
```

- `x`: array of shape `(N, C_in, H, W)`.
- `weight`: array of shape `(C_out, C_in, kh, kw)`.
- `bias`: array of shape `(C_out,)`, or `None` (no bias).
- `stride`, `padding`, `dilation`: each an `int` (same value for height and width) or a pair `(for height, for width)`. Padding adds that many zeros on both sides of the height or width axis.
- Return a float NumPy array `out` of shape `(N, C_out, H_out, W_out)` with $H_{\text{out}} = \lfloor (H + 2p_h - d_h(k_h - 1) - 1)/s_h \rfloor + 1$ (the same for the width), where

$$\text{out}[n, o, i, j] = \text{bias}[o] + \sum_{c, u, v} \text{weight}[o, c, u, v]\; x_{\text{pad}}[n, c,\; i s_h + u d_h,\; j s_w + v d_w].$$

The kernel is not flipped (cross-correlation, as in PyTorch). Do not modify `x`, `weight` or `bias`.

**Example.**

```python
x = np.arange(16.).reshape(1, 1, 4, 4)
w = np.array([[[[1., 0, -1], [1, 0, -1], [1, 0, -1]]]])
conv2d(x, w)                         # [[[[-6., -6.], [-6., -6.]]]]
conv2d(x, w, np.array([0.5]), stride=2, padding=1)
# [[[[-5.5, -3.5], [-26.5, -5.5]]]]
```

**Restrictions.** Only `numpy` (any submodule, including `numpy.lib.stride_tricks`). No Python loops or comprehensions (`for`, `while`, list/dict/set comprehensions, generator expressions). Banned names: `convolve`, `correlate`, `fftconvolve`, `convolve2d`, `correlate2d`.

**Constraints.** $1 \le N \le 8$, $1 \le C_{\text{in}}, C_{\text{out}} \le 32$, $1 \le H, W \le 64$; strides, dilations $\ge 1$; padding $\ge 0$; the output is at least 1 by 1. Time limit 1 s for `x` of shape `(8, 16, 64, 64)` with 32 output channels and a 3 by 3 kernel, CPU.

**Hints.** Lesson u12-l02: `np.pad` with one pair per axis, `sliding_window_view(..., axis=(2, 3))` with the dilated span, slicing with steps for the stride and the dilation, then `np.einsum` or `np.tensordot` over the input channel and the two kernel axes.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem1_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem1_submission.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def conv2d(x, weight, bias=None, stride=1, padding=0, dilation=1):
    # Your code
    return out


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem1_submission as submission
importlib.reload(submission)
# e.g. submission.conv2d(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9W41327iR/1dw2vdqMpUZSfm4rbPeO2/W2aYvXy9JX3vP8uNCIiSzpkiWICN7Vf/v95sBQIKSbMfZ3qVbmwSB+Z7BzADeDNJkcCQGzXhyWI7Gg6EYrFQtMbTZ/VKndaZo7GWRf5kkYlFUa1klIs3Fu2b14Xoo8kJkRVFqmp0prYvcgchGEwae5k2tNEYnz/CapItFOm+y+hojTzGg8rqi58GcUfCSImkyFedyxbjLqphlajWOdTNbpVqnwIFZMsuKtUridFUWVU0IzgZ5syqvB+f4OpN5rhIeJMBF9kXRonlRVSqTNb8sFnX/m3k2RLQz8WoBxobTI1FXjbJDK1mvmiwuSlXJuqjwcSEzTV/BdB3X6UoVTR0z/yMalUtD6jzPI4OxqS1HNITJJc2/kCUTxRxFOkuTNF/G6zRPinU3rtJcN6vuHRRpplYXTTVn6RVVukxzmdGkUlb1xNJ/QwJIM9bMZnAhqxzai0pWxZT/92eVgSst6gtZC6sEsaxkQoNG6iL4dVEVK2GXu9FHv4bRNJ/mJ3a6SLUArzV0/bjJ01o//oF+/fjYQsV7mSY/PjazQUQkXtciUYsUUAUJUiyafE5y0oLMIuHB+JGhrZaXCk9qmncWIowRiV8x9GskTv9+8vbDm9NPIkt1TSwpA5aAaaEvinUusOhjk4vgS6pTUBW+EJ8IXC2qBmjVF1Vd8yKw9kqmWVNhZSVTrQS9inVaXwgpAE/LpRLrKq3BMLmM+EnlR0LLa7EmctdS42uRL+E9BFUk0J24SJME09Mc6tckPaOF3Ip0XpTX7QtZFcnXvrLuBaDmJY1+J14WmZyJlx/+qoWslNDkKBXJKidSDrR4OxEfqmIoNHnvXGYEEfalxaqBUGaKhAoRFeSrKT5nKclhBv4voMqFnMPUI8I0iUakXCmWDek/MJg1YGXqsL6olEyELhUUhllV0SwvsmtYS7YQoAVUwMtI0JUiodciXRCBeGWXtnxYHyKhfPpwevpz/Ork5ef3H8UxYSeGp/k8k8BOaghOr+aqJEsJj6a5wL8Sn8wsGBQkrpIAtpgMxUov3RzgzQsScp7YEfrXKTeguR2QOC+DK7cW4ctb4zRUVPMLb3QB/uGrtcznKrgamu/RZ5Xrogq95YxW1U2Vi6soQVyeXwRhNC8b/GQ1B6GZrJhN8ZrRnVZVUXlQDMserLyMpJZVJa9Bd8cHR5lgWdRDGGVOP2Gfx9MBLBsRejpwlGnIum7KTAXEOeaHES+1tLBMMcdNIljhUCymgw0BvIHCtcElNhpxR12Val7DJjbe/Bug6yibZ4W+i7KhqOoiOx6rw+dDIe3j945eeNYaJLfESl1fg9FFVkgijMYZZ/+Dx8zS8Ecsrc3jLewYBuz0cB9v7hOz1yKAQlKN8JbWWA06siwIBSIFWaH/bW2/+egpksqUI2FOa9J80QK3hkz6zjIrRAjDiot+WHHRD9/wVFVBYiwIIm4lr+iXnOlgKQ7FOgxDgg1G09+UUNjhxIicb4+r9OXEgU58kRnigwgyWS0p7CoyWLHBr6PoyfIm7Ole/bOR2VdYZd+pzFQiOk/Y0ochSxREyLqu7PfpgNXRAdljLp6BdHP+z83CNw2mP7ZyAHE7zuRLtYVAWjnaphjGQNQy8x0MBOMNPv1H1SONZmGop4xUx+wirA560rfqgxF6GumW7NBPoxCFBRxGMSd7cdwjZ4nwR5qzWA/cpAMHtU9prHNZYiev/39Dc1bk6oFBGSidVV6RSfLePgixJSbsvX16giSdQ5CUuIQ+PX3metQRwCBRqjz+jEzPMzBL4GeIb5u8XQBhbwuhoYhg8rernuChlkAOxezb5S6/Su5mivELxvetYpckdoaj+6HAIpIGiQ5m4b5Yg+VdnNm7vI0TM/vIuu17NnEwNNEOlp37unJwGIKXvBTxqoFTILsJFvlQPEI4hWc8enS5pidHiUkdX2IHEIucMS8oQSUekOMgC1zCvWSOlLFaNisk5VTPlZmcq8hlnbzrkz8hLp51niVDzmclLSCM52YmsjPaPPJgmyDzmZakMGQwrENaqpDKUL2kgt/SMjBrGEfPwDmetMZlg8g1ChvYYqIowU8XKWWWlDJ3vHy3ScUfxfjmhVgX1SVl9ZKNV5D6kI+28dIKGcR3EqYkPs0DrSgVBFVfK+UXvogRXf2Ue08mzbUr9kON3FtxWW1y3YN2JZJ3U0MRqvcAUVEJhgJppYklzsPfyrmYAwEtAZMm66ZqhRN7Sg5QXRzOmgSRlIY5B0danlIB1ZRwJ1Gv07kigJBQih26WFgPBs1OoppNiEYWyJdo+yYIWNHgWyQ+04e0wjCYyQ4IBe0LBhlJWCU9s6pHlE0CQoTqchEzGFW5aHOPKSX1/sXIUWqbjlh+j4VVongs/LKhM8mYTLEibwgmvt1BiUDzw7GFtBWKZqhpLrshORmKywmw9eMjkzzcGuxxcr8omE4SxITFMPGGWQyo2IKEtthbxOHnmz5D7Ed1UbDdoPhP6qNosrhBlg8rUy9s1dUaMRsT3I4BiSanen5jRWPXcQ2Jws6peM8/RonKEmXwAqa1pATWFJkbqycGRN5xp4NW+RLeqRKnsK7AgSaTYgVpLyTML24nTrm30mgp0yLmDkOvy/ELjcADQFnXPEg5aFykWUJeN6fOCArmlHyBZv7l0/t3oDVnz9F1AgrZV8vr+gIjh6+FwyN+sC2OOEmrH8UPHQ58xDu88l88Vv+4U/LD2bzyH4q9qrN01g6ZXxiJmjrNuuGiffyHLromQSnrC3+5Tpe57Jbpa73VXnAvFfaFmZxftiOw4pyaBabXoOcp2MxVG+4wcJnWh5nCNDGOnkdjm9PpI+oTrahLOOSQgurCbj4u+JUyhZSX09zhiBZpBot2r8F0ALKLSlEFaDstyESjR3/NL3Pq4Whq4FWi4A4AEKYlaS16ZDPFP59+PIXnWFlEH/A7iGNqg8VxGCFw0XJkdaWsQNk0h1QimhzBWlVVByM4A9IHAhMC4L7eF/eCxHfYq/8pj8Tp09GEO2Fv/nbyP5/in07evTv9GRRspoPYdi7jmJgBY06f5lVdqbl9Qp5vnubFqgSt08HNNH/z/v2H+N37n08/AVwAW4leFfBVevgbDFeZxxN9nc/bD2+QQ74EDPP2SXkvPyPJ7N5+UbnpZ55elaHfYvlsejH3dVliiUpvFZCNNSvEG2xbqvVYLhUdIL8bQZnNPJ5fqPmlDnQ1JxXXsr/ffjT+TtkwNiuSY3FphLPSSwjmBcIiZYCmx8duo+uDVjucDNlWIm1UdZWatmJvizKb0Nn5vmy2rhQSO5YSrAT1NQjt56GfrlGfX3EeSk05tZsfnlmyj2At3Cy21B9RmNS83JbIRW7izEZF9Dsvbo7oGbNR/NxYAm0nnLYEyCtCYIafbLXHSUZn55bQmWzHeZ+s3ZbD7dC40QzMG5/BH2LTK7XjZOJIGeG7aQJXoUysMomCAQwIZn5AFQ0l3emMzgHAziXnbiqLyLAtAu6sezI3bXV/ZMHdEYRqis0Q/lpmlwEpY2vr9tJ0mm7s2ZQG21VFm8rSxIjJPdrdwGxLBflyIHkSJeUrai2Y1wjYIEjKHIMVMk+EIyrmCPiKiTWKCI/2742eLiKZJBZHuDu50403L9JllpK6CefZ4fh8z8JOeVFTJpR171nrF4nZ3WJ8hai3zQ0UCl2xHG3jHex3DrVHlrzCChLPv0OK2xIEtC0peKLrSWCfBWwtvU1437SU+Xyo0E+c72zzvmUQTAc52ldBfedF5LsApsntXO2ftA9jt1lt42THj2RZqjwJXMfHLJoOTMgz4f2/+vLah4V3wZ/S/L3dxU6a5YmmPSg0vZWtBVFh572V9VukjDs2bWLQA4kDYZ5BeiApSbSgbg3/LgswXmJMnkbZIWhe9I+CylMOsIGHJwz3R5c/0o4CwThQm4OhODBAnEeRAx4AH+re5QG1Ym/29xP30f+ZD0J3yNeO9g7Yd+0JIp1RkmznF3aH4FSw2yC4/awkiti/8+/whalGs7W81ofmXBeVDPLCtAVERmmhFXl2PXQ4Ubl0W01grMA0kWwqRUnHtcjUEqX5mtImwTOjKAq9zQxJmUgKUooBfJFyfWm0EHib5h94c/R2YUOu3Xxhhv8SgbeX/kH08kLPhIDhwaYDCrSw8iH4gQ732A4gh79Hw3nhcLRHsp2WQXmf+e3z8X63jaYbJ9tyva9hmCpK2MV/CwebSlbPb0RgMieL4Wx07hv3HvYfIoIO613Mc3DbYZlHv41hkyaxsfcYxTC5MRl1pS5UzkVsmzvyqn8v/5azmzsq9UWVAkx2HXMmGwAECt2Yqql+Rv8+V4dMZ6L0vEq5qhAlTBX+vRT2JJonpO74mgo2uKuieZoP6/sdphm1VVzdGqFuxnMd17NARXHcfkDJZ5NNAo5Mc8Gb+oLwAAQ0taDyU7mcjyq/lgWbla6pZUINK4qy3Ju0RoeflI55WXtviFTBBkOz+TitI98Kk05M+IREeecjlP3fbBjrDc13wl4hEw/aWgyRdrvKpUpWVssvZ22K6Fi5debk3CuJraQoAT8W7ZQnLl2H3WOcGg5RVshEBwET8ZjMhnyCWxEDgoc9izoZgQt2KJ8IoiXGn+Ckwd3EY7HrF6ZgpKGzc3qnUk93r6YxY0zWVtJkiTTwroAKfPhnHbRzIueWUrR1dMpgyQX1maHqnHcLemMvOffTWVfSuXxWc99xB60fJbhrEbA46XKIDsx0f5c3VuJangsQnRbRp5raJ6/fO+GZJk9kfgX27dPrX07efHw7tFW6nYpwFnNz150Ce7Gsf5WJZDkZOVr61TFfgumaVVBnklYKzmeUEYDS7TTL0ajqlFBUjszXn1+/Pf0Yfzw9eTNsqdtKc3SpyHr6rbCIRmPqzZjODnUn+cjE48i7YmZMo2vMQU1eqNrC18x20VlYjJBQB/RjeyE8xszTZ/SZC4ZzY/i7LLEPqSqiHlBsllHkCb9BcKN+b8K2XXotCWuI1jnOTTQzPFIE5qjGEbcuiks+UxAbp4+jaEQdYBe6LJafpFZth8j0QLqW2E9vTkejcUfAg9i4jWJ7MMRk0kkM3RArrKJsEnTHftS69Q7ofYJycYXwwp7JovgkPkAB/nQ0Gh2df7Mj82XIfh+Hh/z7HWaOOyt2J5rgZWg+hfcrt5NVUiiTNJnLdthdCMSNX7c/kIXlw3xyOjCX/sgL2y2jvQfY8m3vEd7ve0vP+Za3+ZKBZmfZ6362seWqPDOFIoO7OuiSeBv2aK+hpCFAobIwgZ2eKLZ/kZV2OKK0Visd8O2ZvL8rcFR15/50gCJBRrAI/YYXogQdQnJKQhi3kmfejsn6qwZxyxgFZSt8iyB3rG0FXArPad4oX39zs8GaeHjEQM6eHZ0jfvO5MKi1XWrB4XE6sKDdZA8bfaVuMJU9+8oUG2/NoQ8NjaLRzUPOxHYvPnzDrvN7dx57OLcblb8ppFklnHWSI0+lxHv37sZOBO8Wm9zD+DgH66zIl+5cZjtmiyDNzS0zzuaHVEAsZCXcyWDoBwGLnA83tnrav5PhlmZKr/fcVXnQXvLvosW/Z8DnFclX7iEdrNbACV4FK06C2w5qh+LJvu3NprPnriCr2sD24K2oXWBy1WMqHYMud70rK+WgMiuKLNghzHwjUHXPcglc3QPXrnAFyq1bCl9rXghX8JjoFsdU39BB2ZE7HqByx5zstpsFnenuuxHdv3/kv0V5HrkuhmTDfnX7od6QD8CH5mrq0L+LM7TXRoZ0Ok0MtLfNsT/YOK+u5KqkeHnuXXS0V93olteaLwPReRR1F0uZ0H3/oUjSTPrHbJSeZ3I1S6SQpozhaoRaH/zCpaThjfdEc2fYu/4rAZOqymOMcZ7//GnYL+JfReYvMQK6SDcUdbDmn7PQkXe8ReXxDrXHLdndpeX2AL/J47nUymQs2PMUJGsFcEw8tGjGHYaRB3vsZGFO6fw7UQBpiW9B3iXT/dcWAbZ3vcvcXWRCUX63N7P4718ETxF8zdIU61jtlethm8DQ3UpQ+xCVe+RRW6a7egpQ/u1TR5r50xH/Gmq7bOeapx1uyTNXhZn17q5oH7B/6fo/u0vXf/J061s6b41WU1ekJ7r8xpdtxs/32CB1Gvg2OrSO/57iPyc5t5jM9wz/xhHMYUQ/DvGIUv/+EfxzRt4zPzpmZ2rFmA4qnRJC4pU0dM+iiQjsCqLZaFJMoNZU6hbG0KN+FD077/xo0hn4eFuKdE0vhzfHCLExwfPluaQtJV8G460LdohA85RuHQ3FBTDDCy637tudBUD6hOULHT6nnQfrQPyzIRH+J/5vTGNPeOA5f/kez2GvQUEaXUZ5Ua24r/CbOnbIGbFfHKx3pxKBNPeSCPSmznamzgt/Z+ypARZq/2aGxoS99ScCJzlifDNPYfoUKDbzAk+XCmE9E5vLG089s4eioGhPMEktHRiKXdtaNIqOrZZj59r7dOmudxEmU1hM2BPGVkET0oHRy8iOTrpRnkwPIzvA7/xl3Cl55GCFfpFBlqNhivDKLWthWu5Ru7GnMVE6uUftT3kq/Xe30p8+WOfW9fivS5xHbsqbLpiKTdJTOXzQcrylMSobEKWaTFbseipZKiZgrwO6vG2/XIxDjdtW2XYY8Vg5EpOrZ9Y820hiVNVxZN4tF1vojLviv2ek9f7HJ63au9/jr6HJugs2FP7DDXqgZry5cBuMr8Z2a7iTJiuC1kHuxWq5z5T8Qn9hRxdlm7zR5hhtC4XZLRCfvt9l23409mbQ89voK7lHjqqoemJ2DW47pIPDZxwbWzwmVD43RHSzjHP4BtGnZmxdfDtslPwXVbv29vR2e4MMaGN9/pT+H3p757ZSJmai74a7Lvhk0ruPa69F8666N8nqGNnKJnYTnnEb1qYDY1bsy1sM3JN1fE/J/83N/wLMZGmA")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
